In [2]:
!pip install langchain-text-splitters
!pip install langchain-huggingface
!pip install langchain-community
!pip install langchain-google-genai

import os

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.5/558.5 kB 14.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [3]:
documents = [

Document(page_content="""
Artificial Intelligence (AI) is the science of building intelligent systems.
Machine Learning is a subset of AI.
Deep Learning is a subset of Machine Learning.
"""),

Document(page_content="""
Natural Language Processing (NLP) enables computers
to understand and generate human language.
Applications include chatbots, translation and sentiment analysis.
"""),

Document(page_content="""
Computer Vision enables machines to recognize images,
objects and videos.
Applications include face recognition and autonomous vehicles.
""")
]

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

# -------------------------------------------------
# STEP 4 : Embeddings
# -------------------------------------------------

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Chunks: 3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
!pip install -U faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.1 MB/s eta 0:00:00


In [7]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

print("FAISS Index Created")

# -------------------------------------------------
# STEP 6 : Retriever
# -------------------------------------------------

retriever = vectorstore.as_retriever(
    search_kwargs={"k":2}
)


FAISS Index Created


In [8]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":2}
)


In [9]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('geminikey')


In [10]:
from google import genai
client=genai.Client(api_key=GEMINI_API_KEY)

In [12]:
import os
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

In [13]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=GEMINI_API_KEY
)


In [14]:
prompt = ChatPromptTemplate.from_template("""

Answer the question only using the context below.

Context:
{context}

Question:
{question}

Answer:

""")

In [15]:
query = "What is Machine Learning?"

In [16]:
retrieved_docs = retriever.invoke(query)

context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

print("\nRetrieved Context\n")
print(context)


Retrieved Context

Artificial Intelligence (AI) is the science of building intelligent systems.
Machine Learning is a subset of AI.
Deep Learning is a subset of Machine Learning.

Computer Vision enables machines to recognize images,
objects and videos.
Applications include face recognition and autonomous vehicles.


In [17]:

final_prompt = prompt.invoke(
    {
        "context":context,
        "question":query
    }
)

# -------------------------------------------------
# STEP 12 : Send to Gemini
# -------------------------------------------------

response = llm.invoke(final_prompt)

print("\nRAG Answer\n")
print(response.content)


RAG Answer

[{'type': 'text', 'text': 'Based on the provided context, Machine Learning is a subset of AI.', 'extras': {'signature': 'EsQICsEIARFNMg/xx1mhuGNLvHOm1mPH6ECmoTKooRhqe2yW7mixKw1E2JjZ+Nc/wXex+AK8pajZFoymz7lyrG+8vhtDUEaK/m1maCEayDEaMiBREXD8aCY70DOSP16p/1Gwjz5mV9RP5pujI6B5Cb7Ga0t5RMYspnNs01m8xNCtjywrANupS2oPO1uGdKqWxwEvt/SPWWYsODAnf5i0AxXJ4xf+/sA1wLt920NpBEziwcWeMkw2lVY9INhJcsG7EfLoTpmItrOU9Jcq7fwTtiuhEO4m/PbWG9Ye0eGvD8M1RyTSsU6+s22UOwRzNTGEvbasHV+RuqfQuzr00sBg6NUwTu0D1i/RuUbYwxBotJDR2LiVtuz5L0P4KpNq5m3ucoTNpYzmJBw0YTyBuNb3kfVsRDRCRojciCNi2eMqneMoWzE4QPyG1Xy2qP3wX72flxwkybndXQFUY7Q8nIAu0xA7FjcgZlsGRs07cThGel8JA80tyWoLOVcaElmS1nmCQ1J/v1C9cTdaBCuLkhPDtml6qfLzRCRENRAKBaO5+97cbkNcI6gh02V4bYFt2DugmEod7j1mZVktrCaxZnsF6+85bv7ETJl9do5mfD5/g/aetP/MYGKqCuTE+XZpMv2d8T9JMTall3n7ZmwfkbMHJC9cjObs6TPpimhx3RCGDyjP3PEMtCWkVDIjnAlp3ApB2v08FdLqOy7Kgb0P1RuFC7K8YjgMuCxHOMYobxtC0xTrXbpkpvw5/CUU8OFCcxiOLsRIvENo0OB69cRGGTOVr6uZZFpFc10VsAN0bZ2CDoI6nJTNef7+Sr+UrXap8LH0qepTLi78PyGbx++j8htsLd/KpmSpB/jkIm

In [18]:
print(response.text)

Based on the provided context, Machine Learning is a subset of AI.
